In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


In [ ]:
# =============================================================================
# Cell 2 - locate the corrected CIC-IDS2017 parquet files.
# Version of record: IEEE CNS 2022 (Liu et al. 2022), acquired as the dhoogla
# Kaggle V5 cleaned-parquet derivative. See Amendment 8 A8.4.
# =============================================================================
CIC17_DIR = config.DATASETS_DIR / 'cicids2017-wtmc2021'   # path name only; not a version claim
CIC17_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(CIC17_DIR.glob('**/*.parquet'))
if not files:
    print('NO DATA in', CIC17_DIR)
    print('Get dhoogla/distrinetcicids2017 (parquet) into this folder, then re-run.')
    raise SystemExit('place the data, then re-run')

print('files:', len(files))
for p in files:
    print('   ', p.name, f'{p.stat().st_size/1e6:.1f} MB')


In [ ]:
# =============================================================================
# Cell 3 - load all day files into one frame; derive capture day.
# =============================================================================
DAYS = ['monday','tuesday','wednesday','thursday','friday']
def derive_day(fname):
    low = fname.lower()
    return next((d for d in DAYS if d in low), 'unknown')

frames = []
for p in files:
    d = pd.read_parquet(p)
    d.columns = [c.strip() for c in d.columns]
    d['source_file'] = p.name
    d['day'] = derive_day(p.name)
    frames.append(d)
cic = pd.concat(frames, ignore_index=True)

lab_col = next((c for c in cic.columns if c.lower() == 'label'), None)
assert lab_col is not None, f'no Label column; columns {list(cic.columns)[:60]}'
cic = cic.rename(columns={lab_col: 'label_raw'})
cic['label_raw'] = cic['label_raw'].astype(str).str.strip()

print('rows:', len(cic), '| columns:', cic.shape[1])
print('days:', cic['day'].value_counts().to_dict())
assert (cic['day'] != 'unknown').all(), 'a file did not encode a weekday'


In [ ]:
# =============================================================================
# Cell 4 (corrected) - label mapping, attempted handling, unit tests.
# Corrections vs the first version (Amendment 8):
#   A8.2  "Infiltration - Portscan" (Thursday) -> Infiltration, NOT PortScan.
#   A8.1  "Attempted-relabel-as-Benign" is Benign; parent family is NOT
#         recoverable in this artifact (no Attempted Category column).
# =============================================================================
def norm(s):
    s = str(s).lower().strip()
    for ch in ['\u2013', '\u2014', '_']:
        s = s.replace(ch, '-')
    s = s.replace('-', ' ')
    while '  ' in s:
        s = s.replace('  ', ' ')
    return s.strip()

FAMILY_MAP = {
    'benign': 'Benign',
    'attempted relabel as benign': 'Benign',    # A8.1: retained as benign; parent lost in this artifact
    'dos hulk': 'DoS', 'dos goldeneye': 'DoS',
    'dos slowloris': 'DoS', 'dos slowhttptest': 'DoS',
    'ddos': 'DDoS',
    'portscan': 'PortScan', 'port scan': 'PortScan',      # Friday standalone PortScan
    'infiltration portscan': 'Infiltration',              # A8.2: Thursday NMAP scan phase of infiltration
    'ftp patator': 'Brute Force', 'ssh patator': 'Brute Force',
    'web attack brute force': 'Web Attack',
    'web attack xss': 'Web Attack',
    'web attack sql injection': 'Web Attack',
    'botnet': 'Bot', 'bot': 'Bot',
    'infiltration': 'Infiltration', 'infilteration': 'Infiltration',
    'heartbleed': 'Heartbleed',
}

SUBTYPE_OVERRIDE = {
    'infiltration portscan': 'Infiltration-NMAP-Portscan',
    'portscan': 'Friday-PortScan',
    'attempted relabel as benign': 'Attempted-relabel-as-Benign',
}

raw = cic['label_raw']
key = raw.map(norm)

unmapped = sorted(set(key[~key.isin(FAMILY_MAP)]))
if unmapped:
    print('UNMAPPED LABELS (add to FAMILY_MAP, then re-run):')
    for u in unmapped:
        print(f'   "{u}"  n={int((key==u).sum())}')
    raise SystemExit('resolve unmapped labels')

cic['label']   = key.map(FAMILY_MAP)
cic['subtype'] = [SUBTYPE_OVERRIDE.get(k, r) for k, r in zip(key, raw)]
cic['attempted_relabelled_benign'] = (key == 'attempted relabel as benign')

# ---- unit tests: the two scans must never collapse and must stay day-pure ----
inf_scan = cic[key == 'infiltration portscan']
fri_scan = cic[key == 'portscan']
assert set(inf_scan['label'].unique()) == {'Infiltration'}, 'infiltration scan misfamilied'
assert set(fri_scan['label'].unique()) == {'PortScan'}, 'Friday portscan misfamilied'
assert set(inf_scan['day'].unique()) <= {'thursday'}, 'infiltration scan not Thursday-only'
assert set(fri_scan['day'].unique()) <= {'friday'}, 'Friday PortScan not Friday-only'
assert (cic.loc[cic['label'] == 'PortScan', 'subtype'] == 'Friday-PortScan').all(), \
       'a non-Friday scan leaked into PortScan'
print('unit tests passed: Infiltration-scan and Friday-PortScan stay distinct and day-pure')

inv = (cic.groupby(['label', 'subtype', 'day']).size()
       .rename('n').reset_index().sort_values(['label', 'n'], ascending=[True, False]))
inv.to_csv(config.REPORTS_DIR / 'cicids2017_label_inventory.csv', index=False)

print('\nfamilies (classes):')
print(cic['label'].value_counts().to_string())
print('\nattempted-relabelled-benign flows:', int(cic['attempted_relabelled_benign'].sum()))


In [ ]:
# =============================================================================
# Cell 5 (corrected) - attempted policy (Amendment 8 A8.1).
#   primary      = retain relabelled flows as Benign (artifact + author guidance)
#   sensitivity  = EXCLUDE the relabelled flows (implementable via the label)
#   withdrawn    = A5.1 merge-to-parent (parent not recoverable here)
# =============================================================================
config.INTERIM_DIR.mkdir(parents=True, exist_ok=True)

cic_primary  = cic.reset_index(drop=True)                                   # nothing excluded
cic_excl_att = cic[~cic['attempted_relabelled_benign']].reset_index(drop=True)

cic_primary.to_parquet(config.INTERIM_DIR / 'cicids2017_primary.parquet', index=False)
cic_excl_att.to_parquet(config.INTERIM_DIR / 'cicids2017_sens_exclude_attempted.parquet', index=False)

defunct = config.INTERIM_DIR / 'cicids2017_attempted_merged.parquet'        # remove the withdrawn artifact
if defunct.exists(): defunct.unlink(); print('removed defunct merged-sensitivity file')

print('PRIMARY (attempted retained as benign):', len(cic_primary), 'rows')
print('  classes:', cic_primary['label'].value_counts().to_dict())
print('SENSITIVITY (attempted excluded):', len(cic_excl_att), 'rows')


In [ ]:
# =============================================================================
# Cell 6 - PROJECTED feasibility preview (non-binding; binding table in nb10).
# =============================================================================
need  = config.min_calib_n(config.ALPHA_PRIMARY)
f_src = config.SPLIT_FRACTIONS['source_cal_pool']

attacks = cic_primary[cic_primary['label'] != 'Benign']
proj = (attacks['label'].value_counts() * f_src).round().astype(int).rename('projected_source_calib').to_frame()
proj['min_calib_needed'] = need
proj['likely_feasible']  = proj['projected_source_calib'] >= need
print(f'whole-dataset projection at alpha={config.ALPHA_PRIMARY}, need >= {need}')
print(proj.to_string())
excluded = proj[~proj['likely_feasible']].index.tolist()
print('likely excluded families:', excluded or 'none')


In [ ]:
# =============================================================================
# Cell 7 - COMPATIBILITY REPORT (Amendment 8 A8.6). Structural record from the
# hashed files, written before any coverage. reports/cicids2017_compatibility.json
# =============================================================================
def counts(s):
    return {str(k): int(v) for k, v in s.value_counts().items()}

report = {
    'version_of_record': 'IEEE CNS 2022 (Liu et al. 2022)',
    'acquired_artifact': 'dhoogla/distrinetcicids2017 Kaggle V5 cleaned parquet',
    'rows_total': int(len(cic)),
    'n_columns': int(cic.shape[1]),
    'attempted_column_present': False,
    'attempted_relabelled_benign_rows': int(cic['attempted_relabelled_benign'].sum()),
    'attempted_parent_recoverable': False,
    'rows_per_day': counts(cic['day']),
    'rows_per_family': counts(cic['label']),
    'rows_per_subtype': counts(cic['subtype']),
    'rows_per_raw_label': counts(cic['label_raw']),
    'feasibility_projection_alpha_0_05': {
        str(k): {'projected_source_calib': int(v),
                 'min_needed': config.min_calib_n(config.ALPHA_PRIMARY),
                 'likely_feasible': bool(v >= config.min_calib_n(config.ALPHA_PRIMARY))}
        for k, v in (cic_primary[cic_primary['label'] != 'Benign']['label']
                     .value_counts() * config.SPLIT_FRACTIONS['source_cal_pool']).round().astype(int).items()
    },
    'excluded_rare_families': excluded,
    'projected_focal_environment': 'DoS within Wednesday (Amendment 7); confirmed in notebook 10',
    'finrst_features_present': [c for c in cic.columns
                                if 'fin' in c.lower() or 'rst' in c.lower()],
}
(config.REPORTS_DIR / 'cicids2017_compatibility.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


In [ ]:
# =============================================================================
# Cell 8 - record/verify file hashes into reports/dataset_hashes.json.
# =============================================================================
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

HP = config.REPORTS_DIR / 'dataset_hashes.json'
H = json.loads(HP.read_text()) if HP.exists() else {}
for p in files:
    k = f'cicids2017/{p.name}'; h = sha256_of(p)
    if k in H:
        print(('OK      ' if H[k] == h else 'MISMATCH'), k)
        if H[k] != h: raise RuntimeError(f'hash changed for {k}')
    else:
        H[k] = h; print('RECORDED', k)
HP.write_text(json.dumps(H, indent=2))


In [ ]:
# =============================================================================
# Cell 9 - dataset manifest, two-layer provenance (Amendment 8 A8.4).
# =============================================================================
mp = config.REPORTS_DIR / 'dataset_manifest.json'
man = json.loads(mp.read_text()) if mp.exists() else {'datasets': {}}
man.setdefault('datasets', {})
man['datasets']['cicids2017'] = {
    'role': 'second environment, corrected CIC-IDS2017',
    'version_of_record': 'IEEE CNS 2022 (Liu et al. 2022)',
    'citation': ('Liu, L., Engelen, G., Lynar, T., Essam, D., Joosen, W. (2022). '
                 'Error Prevalence in NIDS Datasets: A Case Study on CIC-IDS-2017 '
                 'and CSE-CIC-IDS-2018. IEEE CNS 2022, pp. 254-262.'),
    'lineage': 'descends from Engelen et al. (2021), WTMC-2021',
    'acquired_artifact': 'dhoogla/distrinetcicids2017 Kaggle V5 cleaned parquet, 5 per-day files',
    'artifact_transformations': ('type coercion, missing/duplicate removal, parquet conversion, '
                                 'and collapse of per-family X-Attempted labels to a single '
                                 'Attempted-relabel-as-Benign label (not in the official CNS release)'),
    'attempted_policy': ('primary: retain as benign (A8.1); '
                         'sensitivity: exclude relabelled flows; merge-to-parent withdrawn'),
    'taxonomy': ('family = class, variant = subtype (A5.2); '
                 'Infiltration - Portscan -> Infiltration (A8.2)'),
    'families': sorted(cic_primary['label'].unique().tolist()),
    'excluded_rare_families': excluded,
    'class_counts_primary': {k: int(v) for k, v in cic_primary['label'].value_counts().items()},
    'finrst_sensitivity_features': ['FIN Flag Count', 'RST Flag Count', 'Fwd RST Flags', 'Bwd RST Flags'],
    'rows_all': int(len(cic)),
    'n_files': int(len(files)),
}
mp.write_text(json.dumps(man, indent=2))
print(json.dumps(man['datasets']['cicids2017'], indent=2))


In [ ]:
# =============================================================================
# Cell 10 - commit and push. Data stays gitignored; inventory, compatibility
# report, hashes and manifest are committed.
# =============================================================================
def git(*args, show=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r

for s, d in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
             ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, d)

os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb09: correct Infiltration/PortScan taxonomy and attempted-flow handling (Amendment 8)')
    r = git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-4', show=False).stdout)
